# Bước 2 — Bản notebook nhẹ
Upload **step2_colab_light.ipynb** vào Colab, chọn **T4 GPU** rồi chạy từ trên xuống.
Ở ô thứ hai (bootstrap), chọn **step2_bundle.zip** từ máy khi được yêu cầu.
ZIP chứa code và kết quả B01–B05 thật; không cần upload checkpoint B03.

Giữ nguyên 7 lượt train mới T01–T07, ba trục khởi tạo/augmentation/loss.
T00 dùng lại B03 ConvNeXt-Tiny fb_in1k: seed 0, 10 epoch, batch 32, input 224.
Dự trù 2–2,5 giờ trên T4; chỉ train/val, không đánh giá test.
Khi xong tải step2_artifacts.zip, step2_best_checkpoints.zip và notebook có output.


In [ ]:
import sys, subprocess, json, base64, zlib, hashlib
from pathlib import Path
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'timm==1.0.30', 'fvcore==0.1.5.post20221221', 'matplotlib', 'pandas',
    'pillow', 'scikit-learn', 'openpyxl'], capture_output=True, text=True)
print(result.stdout, result.stderr)
result.check_returncode()
BASE_DIR = Path('/content')
PROJECT_DIR = BASE_DIR / 'deepweeds_step0'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Upload ZIP riêng; ô code này không chứa chuỗi base64 khổng lồ.
import zipfile
EXPECTED_BUNDLE_SHA256 = 'ac56d9dd584d91a6c9c68ba68bfb017020421bd6afccb1f13e3cbec894c7cdb8'
BUNDLE_PATH = BASE_DIR / 'step2_bundle.zip'
if not BUNDLE_PATH.exists():
    from google.colab import files
    print('Chọn file step2_bundle.zip trên máy bạn (không chọn .ipynb).')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Chỉ chọn một file step2_bundle.zip.')
    filename, payload = next(iter(uploaded.items()))
    if hashlib.sha256(payload).hexdigest() != EXPECTED_BUNDLE_SHA256:
        raise ValueError('ZIP không khớp notebook. Dùng step2_bundle.zip đi kèm bản nhẹ.')
    BUNDLE_PATH.write_bytes(payload)
    del uploaded, payload
if hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest() != EXPECTED_BUNDLE_SHA256:
    raise ValueError('step2_bundle.zip không khớp notebook; thay bằng ZIP đi kèm.')
sources = {}
with zipfile.ZipFile(BUNDLE_PATH) as bundle:
    for info in bundle.infolist():
        target = (PROJECT_DIR / info.filename).resolve()
        if not target.is_relative_to(PROJECT_DIR.resolve()):
            raise ValueError('Đường dẫn ZIP vượt ngoài thư mục dự án.')
        if not info.is_dir():
            sources[info.filename] = base64.b64encode(bundle.read(info)).decode()
for relative, encoded in sources.items():
    target = PROJECT_DIR / relative
    content = base64.b64decode(encoded)
    if relative == 'results.xlsx' and target.exists():
        continue  # Giữ các sheet/kết quả hiện có.
    if relative.startswith(('runs/','curves/','predictions/','step1/')) and target.exists():
        if target.read_bytes() != content:
            raise FileExistsError(f'Kết quả cũ khác bundle: {target}')
        continue
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
del sources, content
assert hashlib.sha256((PROJECT_DIR / 'eval.py').read_bytes()).hexdigest() == '1210f3c7ca1cb9b4488383f1ceeaa139bfdf5c683694b300f6c55b6b9cdde28e'
CODE_DIR = PROJECT_DIR / 'code'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for name in ['dataset','losses','model','train','step0','step1','step2']:
    sys.modules.pop(name, None)
import dataset, losses, model, train, step0, step1, step2
reference = step2.baseline_record(PROJECT_DIR)
step1.assert_environment(reference)
print('GPU/torch/torchvision/timm khớp B03; giữ công thức nền.')


In [ ]:
command = [sys.executable, '-X', 'utf8', '-c',
    "import torch,unittest; torch.set_num_threads(2); "
    "s=unittest.defaultTestLoader.discover('tests'); "
    "r=unittest.TextTestRunner(verbosity=1).run(s); raise SystemExit(not r.wasSuccessful())"]
result = subprocess.run(command, cwd=PROJECT_DIR, capture_output=True, text=True)
print(result.stdout, result.stderr)
result.check_returncode()


In [ ]:
IMAGES_DIR, LABELS_DIR = step0.prepare_data(BASE_DIR)
frames = dataset.load_split(LABELS_DIR)
split_checks = dataset.check_split(*frames, IMAGES_DIR)
print('CSV fold 0 gốc; test chỉ kiểm tra metadata, không đọc ảnh hay dự đoán.')
from IPython.display import display, Image as DisplayImage


In [ ]:
baseline = step2.run_recipe(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'T00')
step2.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(baseline, ensure_ascii=False, indent=2))
display(DisplayImage(filename=str(PROJECT_DIR / 'curves/T00_convnext_tiny_seed0.png')))


## T01 — A — Khởi tạo từ đầu
Chỉ đổi yếu tố này so với T00.


In [ ]:
row = step2.run_recipe(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'T01')
step2.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))
display(DisplayImage(filename=str(PROJECT_DIR / 'curves/T01_convnext_tiny_seed0.png')))


## T02 — A — Đóng băng backbone, train head
Chỉ đổi yếu tố này so với T00.


In [ ]:
row = step2.run_recipe(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'T02')
step2.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))
display(DisplayImage(filename=str(PROJECT_DIR / 'curves/T02_convnext_tiny_seed0.png')))


## T03 — B — Thêm ColorJitter
Chỉ đổi yếu tố này so với T00.


In [ ]:
row = step2.run_recipe(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'T03')
step2.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))
display(DisplayImage(filename=str(PROJECT_DIR / 'curves/T03_convnext_tiny_seed0.png')))


## T04 — B — CutMix alpha=1
Chỉ đổi yếu tố này so với T00.


In [ ]:
row = step2.run_recipe(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'T04')
step2.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))
display(DisplayImage(filename=str(PROJECT_DIR / 'curves/T04_convnext_tiny_seed0.png')))


## T05 — C — Label smoothing epsilon=0.1
Chỉ đổi yếu tố này so với T00.


In [ ]:
row = step2.run_recipe(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'T05')
step2.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))
display(DisplayImage(filename=str(PROJECT_DIR / 'curves/T05_convnext_tiny_seed0.png')))


## T06 — C — Focal gamma=2
Chỉ đổi yếu tố này so với T00.


In [ ]:
row = step2.run_recipe(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'T06')
step2.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(row, ensure_ascii=False, indent=2))
display(DisplayImage(filename=str(PROJECT_DIR / 'curves/T06_convnext_tiny_seed0.png')))


## T07 — Kiểm tra kết hợp hai yếu tố
Chọn biến thể mới tốt nhất giữa T03/T04 và giữa T05/T06 theo macro-F1 val;
hòa điểm chọn ID trước. Lưu kế hoạch trước khi train. Giữ init=finetune.
Thử kết hợp cả khi hai biến thể mới thua T00 để đo tương tác; không giả định cải thiện.


In [ ]:
plan = step2.plan_combination(PROJECT_DIR)
print(json.dumps(plan, ensure_ascii=False, indent=2))
row = step2.run_recipe(PROJECT_DIR, IMAGES_DIR, LABELS_DIR, 'T07')
step2.export_colab_xlsx(PROJECT_DIR)
display(DisplayImage(filename=str(PROJECT_DIR / 'curves/T07_convnext_tiny_seed0.png')))


In [ ]:
rows = step2.collect_results(PROJECT_DIR)
display(__import__('pandas').DataFrame(rows)[['exp_id','axis','variant','macro_f1_val',
    'delta_f1_pp','top1_val','ece_val','f1_chinee_apple','f1_snake_weed','status']])
selection = step2.write_report(PROJECT_DIR)
step2.export_colab_xlsx(PROJECT_DIR)
print(json.dumps(selection, ensure_ascii=False, indent=2))
print((PROJECT_DIR / 'step2/step2_report.md').read_text(encoding='utf-8'))


In [ ]:
archive = step2.export_artifacts(PROJECT_DIR, BASE_DIR / 'step2_artifacts.zip')
checkpoints = step2.export_checkpoints(PROJECT_DIR, BASE_DIR / 'step2_best_checkpoints.zip')
print('Cần tải cả 2 file:', archive, checkpoints, sep='\n')
print('results.xlsx, report, curves, log/val logits/code đã nằm trong artifacts ZIP.')
print('Checkpoint ZIP giữ đúng đường dẫn T01..T07/seed0/best.pt, tránh trùng tên.')


## Tải kết quả
Chạy riêng hai ô dưới, mỗi ô tải một ZIP. Hoặc tải từ thanh **Files** bên trái.
ZIP checkpoint khoảng 0,8 GB; ZIP kết quả chứa workbook với sheet Backbones và Training.
Tải thêm notebook có output qua **File → Download → Download .ipynb** để lưu bằng chứng.

Nếu muốn resume sau khi phiên bị xóa, cần giữ thêm `last.pt` cùng config/log và `best.pt`.
Khi đã train xong, các `best.pt` đủ cho các thử nghiệm suy luận tiếp theo.


In [ ]:
from google.colab import files
files.download(str(BASE_DIR / 'step2_artifacts.zip'))


In [ ]:
files.download(str(BASE_DIR / 'step2_best_checkpoints.zip'))


Bước 2 hoàn thành khi có đủ T00–T07, đường cong tương ứng, sheet Training,
kết luận từng trục và selection.json. T00 có thể vẫn là tốt nhất.
Một seed/cấu hình chỉ dùng screening; khoảng bootstrap trên val không phải std giữa seed.
Test tiếp tục để riêng đến Bước 4. Sau khi tải, gửi hai ZIP về thư mục dự án để đối chiếu.
